# Lab 2: Julia Basics

**课程**：经济与商务实证研究方法（RMEB） 2026 Spring  
**主题**：Julia 入门与基本使用方法  
**定位**：Week 2 计算环境搭建的配套实验

## 学习目标

完成本 notebook 后，你应该能够：

1. 在 Jupyter 中使用 Julia 内核运行代码。
2. 理解 Julia 的基本语法、变量、数组与字典。
3. 使用 DataFrames 做最基础的数据处理。
4. 固定随机种子并生成随机样本。
5. 画出一个简单的直方图，并理解 Julia 在模拟问题中的优势。

> 这不是一份完整的 Julia 语言手册，而是面向实证研究者的第一份上手材料。

## 为什么在这门课里学 Julia？

在本课程的语言分工里：

- Python 是主力胶水语言。
- Stata 是减少形式因果推断的传统主力。
- R 擅长统计建模和可视化。
- **Julia 适合数值优化、大规模模拟、结构估计加速**。

一句话总结：**Julia 想做的是，把 Python 的易用性和 C 的速度放在一起。**

In [ ]:
using Pkg

required_packages = ["DataFrames", "UnicodePlots"]
for package in required_packages
    try
        @eval using $(Symbol(package))
    catch
        Pkg.add(package)
        @eval using $(Symbol(package))
    end
end

using InteractiveUtils
println("Julia version: ", VERSION)
println("Active project: ", Base.active_project())

Julia version: 1.12.6
Active project: /Users/happyhome/.julia/environments/v1.12/Project.toml


## 1. Julia 基本语法

Julia 的语法对 Python 用户相对友好，但也有一些自己的风格：

- 用 `=` 赋值。
- 函数可以写成多行，也可以写成单行。
- 字符串插值用 `$变量名`。
- 很多函数名后面带 `!`，表示**这个函数会原地修改对象**。

In [ ]:
course_name = "RMEB"
week = 2
temperature = 23.5

println("课程：$course_name")
println("第 $week 周：Julia 入门")
println("今天教室温度约为 $temperature 度。")

square(x) = x^2
println("5 的平方是 ", square(5))

课程：RMEB
第 2 周：Julia 入门
今天教室温度约为 23.5 度。
5 的平方是 25


## 2. 数组、向量与字典

Julia 的数值计算非常依赖数组操作。先熟悉最常见的几种对象：

- 向量：一维数组
- 矩阵：二维数组
- 字典：键值对结构

In [ ]:
scores = [88, 92, 79, 95, 90]
X = [1 2; 3 4; 5 6]
student = Dict("name" => "Alice", "group" => 3, "treated" => true)

println("scores = ", scores)
println("scores 的平均值 = ", sum(scores) / length(scores))
println("X = ")
display(X)
println("student[\"name\"] = ", student["name"])

scores = [88, 92, 79, 95, 90]
scores 的平均值 = 88.8
X = 
student["name"] = Alice


3×2 Matrix{Int64}:
 1  2
 3  4
 5  6

## 3. 向量化与广播

Julia 的一个常见写法是广播（broadcasting），也就是在运算符或函数前后加点：

- `x .^ 2`：对向量中每个元素平方
- `log.(x)`：对每个元素取对数

这在模拟和矩阵运算里非常常见。

In [ ]:
x = [1, 2, 3, 4, 5]
x_squared = x .^ 2
x_shifted = x .+ 10

println("原向量 x = ", x)
println("逐元素平方后 = ", x_squared)
println("每个元素加 10 后 = ", x_shifted)

原向量 x = [1, 2, 3, 4, 5]
逐元素平方后 = [1, 4, 9, 16, 25]
每个元素加 10 后 = [11, 12, 13, 14, 15]


## 4. DataFrames：最基本的数据表操作

做实证研究时，我们经常处理表格型数据。Julia 中最常用的包之一是 `DataFrames.jl`。

下面做一个极小的示例，模拟一个面板数据切片。

In [ ]:
using DataFrames
using Statistics

df = DataFrame(
    id = [1, 1, 2, 2, 3, 3],
    year = [2024, 2025, 2024, 2025, 2024, 2025],
    treatment = [0, 1, 0, 0, 1, 1],
    outcome = [10.2, 11.5, 9.8, 10.0, 12.1, 12.7]
)

println("完整数据：")
display(df)

println("\n只看 2025 年：")
display(filter(row -> row.year == 2025, df))

df.outcome_gap = df.outcome .- mean(df.outcome)
println("\n加入 outcome_gap 后：")
display(df)

Row,id,year,treatment,outcome
,Int64,Int64,Int64,Float64
1,1,2024,0,10.2
2,1,2025,1,11.5
3,2,2024,0,9.8
4,2,2025,0,10.0
5,3,2024,1,12.1
6,3,2025,1,12.7


完整数据：

只看 2025 年：

加入 outcome_gap 后：


Row,id,year,treatment,outcome
,Int64,Int64,Int64,Float64
1,1,2025,1,11.5
2,2,2025,0,10.0
3,3,2025,1,12.7


Row,id,year,treatment,outcome,outcome_gap
,Int64,Int64,Int64,Float64,Float64
1,1,2024,0,10.2,-0.85
2,1,2025,1,11.5,0.45
3,2,2024,0,9.8,-1.25
4,2,2025,0,10.0,-1.05
5,3,2024,1,12.1,1.05
6,3,2025,1,12.7,1.65


## 5. 随机数、种子与可重复性

Monte Carlo 模拟的核心是：

1. 明确数据生成过程（DGP）
2. 固定随机种子
3. 反复抽样与汇总

先从最简单的标准正态随机样本开始。

In [ ]:
using Random
using Statistics

Random.seed!(42)
z = randn(10)

println("10 个标准正态随机数：")
println(z)
println("均值 = ", round(mean(z), digits=4))
println("标准差 = ", round(std(z), digits=4))

10 个标准正态随机数：
[-0.36335748145177754, 0.2517372155742292, -0.31498797116895605, -0.31125240132442067, 0.8163067649323273, 0.47673837983187795, -0.8595553820616212, -1.4692882055065464, -0.20661311448119266, -0.31074387308373413]
均值 = -0.2291
标准差 = 0.6495


## 6. 一个简单直方图

这里我们生成 1000 个标准正态随机数，并用 `UnicodePlots` 画一个文本直方图。

这不是发表级图形，但足够用于理解分布形状，且依赖轻、运行稳。

In [ ]:
using UnicodePlots

Random.seed!(42)
draws = randn(1000)

println("n = ", length(draws))
println("mean = ", round(mean(draws), digits=4))
println("std = ", round(std(draws), digits=4))
println()
println(histogram(draws; nbins=30, title="标准正态样本直方图", xlabel="x", ylabel="count"))

n = 1000
mean = -0.0601
std = 0.9872

                                       标准正态样本直方图                 
                      ┌                                        ┐ 
         [-3.0, -2.8) ┤▍ 1                                       
         [-2.8, -2.6) ┤█▋ 4                                      
         [-2.6, -2.4) ┤█▎ 3                                      
         [-2.4, -2.2) ┤████▌ 11                                  
         [-2.2, -2.0) ┤██▊ 7                                     
         [-2.0, -1.8) ┤█████▍ 13                                 
         [-1.8, -1.6) ┤██████████▋ 26                            
         [-1.6, -1.4) ┤████████████▋ 31                          
         [-1.4, -1.2) ┤███████████▊ 29                           
         [-1.2, -1.0) ┤████████████████▊ 41                      
         [-1.0, -0.8) ┤█████████████████████████████▌ 72         
         [-0.8, -0.6) ┤██████████████████████▎ 54                
         [-0.6, -0.4) ┤███████████████

## 4. 循环与控制流

Julia 的循环语法简洁，且提供列表推导式（comprehension）来写更紧凑的代码。

## 4. 循环与控制流

Julia 的循环语法简洁，且提供列表推导式（comprehension）来写更紧凑的代码。

In [ ]:
# --- for 循环 ---
println("=== for 循环 ===")
for i in 1:5
    println("i = $i")
end

# 遍历数组
fruits = ["苹果", "香蕉", "橙子"]
for (idx, fruit) in enumerate(fruits)
    println("第$(idx)个: $fruit")
end

# --- while 循环 ---
println("\n=== while 循环 ===")
n, total = 1, 0
while n <= 10
    total += n
    n += 1
end
println("1到10的和 = $total")

# --- 列表推导式 (comprehension) ---
println("\n=== 推导式 ===")
squares = [x^2 for x in 1:10]
println("平方数: $squares")

# 带条件的推导式
evens = [x for x in 1:20 if x % 2 == 0]
println("偶数: $evens")

# 二维推导式（矩阵生成）
matrix = [i + j for i in 1:3, j in 1:4]
display(matrix)

# --- if/elseif/else ---
score = 85
grade = if score >= 90
    "优秀"
elseif score >= 80
    "良好"
else
    "需努力"
end
println("\n成绩: $grade")

# 三元运算符
x = -3
println("$x 是 $(x > 0 ? "正数" : "非正数")")

## 5. 函数的写法

Julia 函数非常灵活：支持单行定义、多返回值、可选参数、关键字参数，以及强大的**多重派发**（Multiple Dispatch）。

In [ ]:
# --- 基本函数 ---
function greet(name)
    return "你好，$(name)！"
end
println(greet("同学"))

# 单行简写
square(x) = x^2
println("5² = $(square(5))")

# --- 多返回值 ---
function stats(data)
    μ = sum(data) / length(data)
    σ = sqrt(sum((x - μ)^2 for x in data) / (length(data) - 1))
    return μ, σ
end

mean_val, std_val = stats([2, 4, 6, 8, 10])
println("均值 = $mean_val, 标准差 = $(round(std_val, digits=3))")

# --- 默认参数与关键字参数 ---
function power(base, exp=2; verbose=false)
    result = base^exp
    verbose && println("$base^$exp = $result")
    return result
end

power(3)           # 默认 exp=2
power(2, 10)       # 指定 exp=10
power(2, 3; verbose=true)  # 关键字参数

# --- 匿名函数 ---
double = x -> 2x
println("double(7) = $(double(7))")

# 与 map/filter 配合
println("平方: ", map(x -> x^2, 1:5))
println("偶数: ", filter(x -> x % 2 == 0, 1:10))

# --- 多重派发 (Multiple Dispatch) ---
# Julia 的核心特性：同名函数根据参数类型自动选择方法
describe(x::Int) = "整数: $x"
describe(x::Float64) = "浮点数: $(round(x, digits=3))"
describe(x::String) = "字符串: \"$x\" (长度=$(length(x)))"
describe(x::Vector) = "向量: 长度=$(length(x)), 类型=$(eltype(x))"

println(describe(42))
println(describe(3.14159))
println(describe("你好"))
println(describe([1, 2, 3]))

## 6. 高级用法入门

### 6.1 类型系统与性能

Julia 的类型系统是其高性能的秘密——正确使用类型可以让代码接近 C 的速度。

### 6.2 宏 (Macros)

宏是 Julia 的元编程工具，可以在编译时生成代码。内置宏如 `@time`、`@show`、`@assert` 非常实用。

### 6.3 广播与向量化

Julia 用 `.` 运算符实现逐元素操作，比手写循环更简洁。

In [ ]:
# ===== 6.1 类型系统与性能 =====
# 类型注解（可选但利于性能）
function fibonacci(n::Int)::Int
    n <= 1 && return n
    a, b = 0, 1
    for _ in 2:n
        a, b = b, a + b
    end
    return b
end
println("fib(10) = $(fibonacci(10))")

# 自定义类型 (struct)
struct Student
    name::String
    score::Float64
end

s = Student("张三", 92.5)
println("$(s.name) 的成绩: $(s.score)")

# @time 宏：测量执行时间和内存分配
@time sum(rand(1_000_000))

# ===== 6.2 常用宏 =====
# @show：调试利器
x = 42
@show x^2 + 1

# @assert：断言检查
@assert 1 + 1 == 2 "数学出了问题！"

# @elapsed：只返回时间（秒）
t = @elapsed begin
    total = 0.0
    for i in 1:1_000_000
        total += sin(i)
    end
end
println("耗时: $(round(t, digits=4)) 秒")

# ===== 6.3 广播（点运算符）=====
# 逐元素操作：函数名后加 .
A = [1, 2, 3, 4, 5]
println("sin.(A) = ", round.(sin.(A), digits=3))
println("A.^2 = ", A.^2)

# 等价于 map(sin, A)，但更通用
B = [1 2; 3 4]
println("矩阵逐元素 +10:")
display(B .+ 10)

# @. 宏：自动给所有操作加点
result = @. sin(A)^2 + cos(A)^2  # 应该全是 1.0
println("sin²+cos² = ", result)

## ✨ Julia 杀手锏预览：C 的速度 + Python 的优雅

Julia 兼具高级语言的表达力和编译型语言的速度。在大规模模拟、Bootstrap、Monte Carlo 中，Julia 比 R/Python 快 10–100 倍。下面用 10,000 次 Monte Carlo 模拟验证 OLS 的无偏性。

> 💡 **研究中的组合优势**：Julia 做大规模模拟/Bootstrap + Stata 做面板回归 + Python 做 ML + R 做可视化 = 完整研究工具链。

In [ ]:
using Random, Statistics

# Julia 的杀手锏：Monte Carlo — 像写数学公式一样，跑起来像 C
function monte_carlo_ols_bias(n, β_true, n_sim)
    bias = zeros(length(β_true))
    for s in 1:n_sim
        X = randn(n, length(β_true))
        y = X * β_true + randn(n)
        β_hat = X \ y          # OLS: 一个反斜杠搞定
        bias .+= β_hat .- β_true
    end
    return bias ./ n_sim
end

# 1,000,000 次 Monte Carlo 模拟 — Julia 几秒完成（R/Python 可能要几分钟）
Random.seed!(42)
β_true = [2.0, -1.5, 0.8]
@time bias = monte_carlo_ols_bias(100, β_true, 1_000_000)

println("\nOLS 无偏性验证 (n=100, 1,000,000次模拟):")
for (i, b) in enumerate(bias)
    println("  β$i 平均偏差 = $(round(b, digits=4))")
end
println("\n💡 偏差 ≈ 0 → OLS 是无偏估计量！Julia 让 Monte Carlo 验证理论变得轻而易举")

  4.965791 seconds (32.44 M allocations: 55.859 GiB, 22.87% gc time, 9.05% compilation time)

OLS 无偏性验证 (n=100, 1,000,000次模拟):
  β1 平均偏差 = -0.0
  β2 平均偏差 = 0.0001
  β3 平均偏差 = -0.0001

💡 偏差 ≈ 0 → OLS 是无偏估计量！Julia 让 Monte Carlo 验证理论变得轻而易举


### 🐍 对比：同样的模拟用 Python 要多久？

下面这个 cell 使用 **Python 内核**运行完全相同的 Monte Carlo 模拟。运行前请在 VS Code 中将此 cell 的内核切换为 Python（点击 cell 右下角的语言标识）。

对比上方 Julia `@time` 的结果，你会直观感受到 Julia 在数值计算中的速度优势。

In [1]:
import numpy as np
import time

# 同样的 Monte Carlo OLS 模拟 — Python 版本
def monte_carlo_ols_bias_python(n, beta_true, n_sim):
    beta_true = np.array(beta_true)
    bias = np.zeros(len(beta_true))
    for s in range(n_sim):
        X = np.random.randn(n, len(beta_true))
        y = X @ beta_true + np.random.randn(n)
        beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]  # OLS
        bias += beta_hat - beta_true
    return bias / n_sim

# 100,000 次 Monte Carlo 模拟 — 对比 Julia 的速度
np.random.seed(42)
beta_true = [2.0, -1.5, 0.8]

start = time.time()
bias = monte_carlo_ols_bias_python(100, beta_true, 1_000_000)
elapsed = time.time() - start

print(f"Python 用时: {elapsed:.4f} 秒")
print(f"\nOLS 无偏性验证 (n=100, 1,000,000次模拟):")
for i, b in enumerate(bias):
    print(f"  β{i+1} 平均偏差 = {b:.4f}")
print(f"\n⬆️ 对比上方 Julia 的 @time 结果，感受速度差异！")

Python 用时: 18.2636 秒

OLS 无偏性验证 (n=100, 1,000,000次模拟):
  β1 平均偏差 = 0.0001
  β2 平均偏差 = 0.0002
  β3 平均偏差 = 0.0001

⬆️ 对比上方 Julia 的 @time 结果，感受速度差异！


## 7. 小结

你现在已经见过 Julia 在研究工作流里的几个基础模块：

- 环境与包管理
- 变量与函数
- 向量和广播
- DataFrame 表格操作
- 随机数与可重复性
- 简单直方图

对于本课程而言，Julia 不是替代所有语言，而是在**需要数值优化和大规模模拟时，提供更高性能的工具**。

## 8. 练习

请完成下面几个小练习。建议先复制一份代码，再修改。

In [ ]:
# TODO 1
# 生成 20 个均值为 5、标准差为 2 的正态随机数，
# 然后计算它们的样本均值和样本标准差。

# 在这里写你的代码

In [ ]:
# TODO 2
# 基于上面的 df，创建一个新变量 treated_outcome，
# 令它等于 outcome * treatment。

# 在这里写你的代码

In [ ]:
# TODO 3
# 把直方图示例改成：从 Uniform(0,1) 分布抽取 1000 个数，
# 然后比较它和标准正态直方图的形状差别。

# 在这里写你的代码